In [1]:
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
  raise ValueError("HF_TOKEN not found.Check Colab Secrets.")
print("Hugging Face token loaded successfully!")

Hugging Face token loaded successfully!


In [4]:
!pip -q install -U huggingface_hub

In [5]:
from huggingface_hub import InferenceClient

client = InferenceClient(
    api_key=HF_TOKEN,
    provider="auto"
)

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Explain AWS S3 in two simple sentences."
        }
    ],
    max_tokens=120
)

print(response.choices[0].message.content)

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6ad841b0-334c03d7052b97487bf3da8c;ea89ba86-3f89-400e-9b51-661cfd06abfb)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have no remaining credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.

In [1]:
!pip install google-auth==2.49.0

In [3]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Explain AWS S3 in two simple sentences."
)

print(response.text)

AWS S3 is a cloud storage service that acts like an unlimited, secure hard drive on the internet. It allows you to easily store, back up, and retrieve any type of file from anywhere at any time.


In [4]:
!pip -q install chromadb sentence-transformers requests beautifulsoup4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/13

In [5]:
import requests
from bs4 import BeautifulSoup

urls = {
    "Amazon S3": "https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html",
    "Amazon EC2": "https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/concepts.html",
    "AWS IAM": "https://docs.aws.amazon.com/IAM/latest/UserGuide/introduction.html"
}

documents = []

for service, url in urls.items():
    try:
        response = requests.get(
            url,
            timeout=30,
            headers={"User-Agent": "AWS-RAG-Capstone/1.0"}
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()

        text = soup.get_text(separator=" ", strip=True)

        if len(text) > 200:
            documents.append({
                "source": service,
                "url": url,
                "text": text
            })
            print(f"{service}: downloaded {len(text)} characters")
        else:
            print(f"{service}: insufficient text extracted")

    except requests.RequestException as e:
        print(f"{service}: download failed - {e}")

print("\nTotal documents collected:", len(documents))

Amazon S3: downloaded 36631 characters
Amazon EC2: downloaded 10408 characters
AWS IAM: downloaded 5149 characters

Total documents collected: 3


In [6]:
chunks = []
chunk_size = 1000
overlap = 150

for doc in documents:
    text = doc["text"]
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end]

        if chunk_text.strip():
            chunks.append({
                "text": chunk_text,
                "source": doc["source"],
                "url": doc["url"]
            })

        start += chunk_size - overlap

print("Total chunks created:", len(chunks))
print("\nFirst chunk preview:\n")
print(chunks[0]["text"][:500] if chunks else "No chunks created")

Total chunks created: 64

First chunk preview:

What is Amazon S3? - Amazon Simple Storage Service View a markdown version of this page What is Amazon S3? - Amazon Simple Storage Service Documentation Amazon Simple Storage Service (S3) User Guide Features of Amazon S3 How Amazon S3 works Amazon S3 data consistency model Related services Accessing Amazon S3 Paying for Amazon S3 PCI DSS compliance What is Amazon S3? Amazon Simple Storage Service (Amazon S3) is an object storage service that offers industry-leading scalability,
		data availabili


In [7]:
import chromadb
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

client_db = chromadb.Client()

collection = client_db.get_or_create_collection(
    name="aws_support_knowledge"
)

texts = [chunk["text"] for chunk in chunks]
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
).tolist()

collection.add(
    ids=[f"aws_chunk_{i}" for i in range(len(chunks))],
    documents=texts,
    embeddings=embeddings,
    metadatas=[
        {"source": chunk["source"], "url": chunk["url"]}
        for chunk in chunks
    ]
)

print("Chunks stored:", collection.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Chunks stored: 64


In [9]:
question = "What is Amazon S3 used for?"

query_embedding = embedding_model.encode(
    [question]
).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

for i, text in enumerate(results["documents"][0]):
    metadata = results["metadatas"][0][i]

    print(f"\nResult {i + 1}")
    print("Source:", metadata["source"])
    print("URL:", metadata["url"])
    print("Content:", text[:400])


Result 1
Source: Amazon S3
URL: https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html
Content: What is Amazon S3? - Amazon Simple Storage Service View a markdown version of this page What is Amazon S3? - Amazon Simple Storage Service Documentation Amazon Simple Storage Service (S3) User Guide Features of Amazon S3 How Amazon S3 works Amazon S3 data consistency model Related services Accessing Amazon S3 Paying for Amazon S3 PCI DSS compliance What is Amazon S3? Amazon Simple Storage Service 

Result 2
Source: Amazon S3
URL: https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html
Content: GET, HEAD, and LIST requests to modify and process data as
						it is returned to an application. Filter rows, dynamically resize images,
						redact confidential data, and much more. Event
							notifications â Trigger workflows that use Amazon Simple Notification Service
						(Amazon SNS), Amazon Simple Queue Service (Amazon SQS), and AWS Lambda when a change is made to 

In [21]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=api_key)


def ask_aws_assistant(question):
    # 1. Convert the question into an embedding
    query_embedding = embedding_model.encode(
        [question]
    ).tolist()

    # 2. Retrieve the three most relevant chunks
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3
    )

    # 3. Build the context and source list
    context_parts = []
    sources = []

    for text, metadata in zip(
        results["documents"][0],
        results["metadatas"][0]
    ):
        context_parts.append(text)

        if metadata["url"] not in sources:
            sources.append(metadata["url"])

    context = "\n\n".join(context_parts)

    # 4. Ask Gemini to answer using retrieved evidence
    prompt = f"""
You are an AWS Cloud Support Assistant.

Answer the user's question using only the supplied context.
If the context does not contain enough information, say so.
Do not invent AWS configuration details.

Context:
{context}

Question:
{question}

Provide a clear, beginner-friendly answer.
"""

    response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
    )

    return response.text, sources


answer, sources = ask_aws_assistant(
    "What is Amazon S3 used for?"
)

print("ANSWER:\n")
print(answer)

print("\nSOURCES:")
for source in sources:
    print(source)

ANSWER:

Based on the provided context, Amazon S3 (Amazon Simple Storage Service) is used to store and protect any amount of data for a wide range of use cases. These include:

* Data lakes
* Websites
* Mobile applications
* Backup and restore
* Archive
* Enterprise applications
* IoT (Internet of Things) devices
* Big data analytics

SOURCES:
https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html


In [29]:

import time

MODELS_TO_TRY = ["gemini-3.5-flash-lite"]

def ask_aws_assistant(question):
    # 1. Retrieve relevant AWS documentation
    query_embedding = embedding_model.encode(
        [question]
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3,
        include=["documents", "metadatas", "distances"]
    )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    # 2. Filter documents using the distance threshold
    context_parts = []
    sources = []

    for doc, metadata, distance in zip(
        documents, metadatas, distances
    ):
        if distance < 1.2:
            context_parts.append(doc)

            source = metadata.get("url") or metadata.get("source")

            if source and source not in sources:
                sources.append(source)

    # 3. Stop if no relevant documents were found
    if not context_parts:
        return (
            "I couldn't find sufficiently relevant information "
            "in the available AWS documentation.",
            []
        )

    # 4. Prepare the prompt
    context = "\n\n".join(context_parts)

    prompt = f"""
You are an AWS Cloud Support Assistant.

Answer using only the documentation provided.
Use simple, beginner-friendly language.
If the context does not contain enough information,
clearly say so. Do not invent technical details.

DOCUMENTATION:
{context}

USER QUESTION:
{question}
"""

    # 5. Generate the answer with retry handling
    for model_name in MODELS_TO_TRY:
        for attempt in range(2):
            try:
                response = gemini_client.models.generate_content(
                    model=model_name,
                    contents=prompt
                )

                if response.text:
                    return response.text, sources

            except Exception as e:
                print(
                    f"{model_name}, attempt {attempt + 1}: "
                    f"{type(e).__name__}"
                )

                error_text = str(e)

                if "503" in error_text or "UNAVAILABLE" in error_text:
                    time.sleep(2 * (attempt + 1))
                else:
                    break

    return (
        "The language model is temporarily unavailable. "
        "Please try again later.",
        sources
    )


In [30]:
answer, sources = ask_aws_assistant("What is Amazon S3 used for?")

print("AWS QUESTION")
print("Answer:", answer)
print("Sources:", sources)

print("\n" + "=" * 50 + "\n")

answer, sources = ask_aws_assistant("Who won the FIFA World Cup?")

print("UNRELATED QUESTION")
print("Answer:", answer)
print("Sources:", sources)

AWS QUESTION
Answer: Based on the provided documentation, Amazon S3 is used to store and protect any amount of data for a range of use cases, such as:

* Data lakes
* Websites
* Mobile applications
* Backup and restore
* Archive
* Enterprise applications
* IoT devices
* Big data analytics
Sources: ['https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html']


UNRELATED QUESTION
Answer: I couldn't find sufficiently relevant information in the available AWS documentation.
Sources: []


In [12]:
from google.colab import userdata
from google import genai

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

print("Models available for text generation:\n")

for model in client.models.list():
    if "generateContent" in (model.supported_actions or []):
        print(model.name)

Models available for text generation:

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-prev

In [13]:
from google.colab import userdata
from google import genai

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Explain AWS S3 in two simple sentences."
)

print(response.text)

AWS S3 is a cloud-based storage service that allows you to securely store, access, and back up any amount of data, like files and images, over the internet. It works like an infinitely expandable digital filing cabinet that you can easily manage from anywhere.


In [17]:
ask_aws_assistant("What is Amazon S3 used for?")

('Based on the provided context, Amazon S3 (Amazon Simple Storage Service) is used to store and protect any amount of data for a wide range of use cases, including:\n\n* Data lakes\n* Websites\n* Mobile applications\n* Backup and restore\n* Archive\n* Enterprise applications\n* IoT (Internet of Things) devices\n* Big data analytics',
 ['https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html'])

In [18]:
!pip install -q gradio

In [19]:

import gradio as gr

def chat_with_aws(question, history):
    if not question.strip():
        return "Please enter an AWS question."

    try:
        answer, sources = ask_aws_assistant(question)

        # Remove duplicate sources
        unique_sources = list(set(sources))

        source_text = "\n".join(unique_sources)

        return f"{answer}\n\n**Sources:**\n{source_text}"

    except Exception as e:
        return f"Sorry, an error occurred: {str(e)}"


demo = gr.ChatInterface(
    fn=chat_with_aws,
    title="AWS Cloud Support Assistant",
    description=(
        "Ask questions about Amazon S3, Amazon EC2, "
        "AWS IAM, and other topics covered by the loaded documentation."
    ),
    examples=[
        "What is Amazon S3 used for?",
        "What is Amazon EC2?",
        "What is AWS IAM used for?",
        "How can I protect data stored in Amazon S3?"
    ]
)

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6fcfd6f7f9266ce754.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6fcfd6f7f9266ce754.gradio.live


In [22]:
answer, sources = ask_aws_assistant(
    "What is Amazon S3 used for?"
)

print("ANSWER:\n")
print(answer)

print("\nSOURCES:")
for source in sources:
    print(source)

ANSWER:

Based on the provided context, Amazon S3 (Amazon Simple Storage Service) is used to store and protect any amount of data for a wide range of use cases, including:

* Data lakes
* Websites
* Mobile applications
* Backup and restore
* Archive
* Enterprise applications
* IoT devices
* Big data analytics

SOURCES:
https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html


In [23]:
answer, sources = ask_aws_assistant(
    "Who won the FIFA World Cup?"
)

print("ANSWER:\n")
print(answer)

print("\nSOURCES:")
for source in sources:
    print(source)

ANSWER:

The provided context does not contain enough information to answer who won the FIFA World Cup.

SOURCES:
https://docs.aws.amazon.com/IAM/latest/UserGuide/introduction.html
https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html


In [24]:
demo.close()

demo = gr.ChatInterface(
    fn=chat_with_aws,
    title="AWS Cloud Support Assistant",
    description=(
        "Ask questions about AWS using retrieved documentation "
        "from Amazon S3, Amazon EC2, and AWS IAM."
    ),
    examples=[
        "What is Amazon S3 used for?",
        "What is Amazon EC2?",
        "How does AWS IAM manage access?",
        "Who won the FIFA World Cup?"
    ]
)

demo.launch(share=True, debug=True)

Closing server running on port: 7860
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3c8a326dba97de57c6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3c8a326dba97de57c6.gradio.live


In [25]:

import pandas as pd
from IPython.display import display

test_questions = [
    {
        "question": "What is Amazon S3 used for?",
        "expected_topic": "S3",
        "expected_answer": "Store and retrieve data",
        "should_have_source": True
    },
    {
        "question": "What is Amazon EC2?",
        "expected_topic": "EC2",
        "expected_answer": "Virtual servers in the cloud",
        "should_have_source": True
    },
    {
        "question": "What is AWS IAM used for?",
        "expected_topic": "IAM",
        "expected_answer": "Manage access and permissions",
        "should_have_source": True
    },
    {
        "question": "How can I back up data in Amazon S3?",
        "expected_topic": "S3",
        "expected_answer": "Backup and restore data",
        "should_have_source": True
    },
    {
        "question": "Who won the FIFA World Cup?",
        "expected_topic": "OUT_OF_SCOPE",
        "expected_answer": "Context is insufficient",
        "should_have_source": False
    }
]

test_df = pd.DataFrame(test_questions)

display(test_df)


,question,expected_topic,expected_answer,should_have_source
0,What is Amazon S3 used for?,S3,Store and retrieve data,True
1,What is Amazon EC2?,EC2,Virtual servers in the cloud,True
2,What is AWS IAM used for?,IAM,Manage access and permissions,True
3,How can I back up data in Amazon S3?,S3,Backup and restore data,True
4,Who won the FIFA World Cup?,OUT_OF_SCOPE,Context is insufficient,False


In [26]:

evaluation_results = []

for item in test_questions:
    print("Testing:", item["question"])

    answer, sources = ask_aws_assistant(item["question"])

    evaluation_results.append({
        "Question": item["question"],
        "Answer": answer,
        "Sources Found": len(sources),
        "Source Check": (
            "PASS" if sources else "NO SOURCE"
        )
    })

    print("Answer:", answer)
    print("Sources:", sources)
    print("-" * 60)

evaluation_df = pd.DataFrame(evaluation_results)

display(
    evaluation_df[
        ["Question", "Sources Found", "Source Check"]
    ]
)


Testing: What is Amazon S3 used for?
Answer: Based on the provided context, Amazon S3 (Amazon Simple Storage Service) is used to store and protect any amount of data for a wide range of use cases, including:

* Data lakes
* Websites
* Mobile applications
* Backup and restore
* Archive
* Enterprise applications
* IoT devices
* Big data analytics
Sources: ['https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html']
------------------------------------------------------------
Testing: What is Amazon EC2?
Answer: Amazon Elastic Compute Cloud (Amazon EC2) is a service that provides on-demand, scalable computing capacity in the AWS Cloud. 

In simple terms, an EC2 instance is a virtual server in the AWS Cloud that allows you to:
* Launch as many or as few virtual servers as you need.
* Configure security and networking, and manage storage.
* Easily add capacity (scale up) to handle heavy tasks or traffic spikes, and reduce capacity (scale down) when usage decreases. 

Using Amazon 

,Question,Sources Found,Source Check
0,What is Amazon S3 used for?,1,PASS
1,What is Amazon EC2?,1,PASS
2,What is AWS IAM used for?,1,PASS
3,How can I back up data in Amazon S3?,1,PASS
4,Who won the FIFA World Cup?,2,PASS


In [27]:

def check_evaluation(row):
    question = row["Question"].lower()
    answer = row["Answer"].lower()
    sources_found = row["Sources Found"]

    if "fifa world cup" in question:
        # Out-of-scope question should be declined
        if (
            "not contain enough information" in answer
            or "insufficient" in answer
            or "couldn't find" in answer
            or "could not find" in answer
        ):
            return "PASS - Out-of-scope handled"
        return "CHECK - Expected insufficient-context response"

    # In-scope AWS questions should have sources
    if sources_found > 0:
        return "PASS - Source returned"

    return "CHECK - No source returned"


evaluation_df["Evaluation Check"] = evaluation_df.apply(
    check_evaluation, axis=1
)

display(
    evaluation_df[
        ["Question", "Sources Found", "Evaluation Check"]
    ]
)


,Question,Sources Found,Evaluation Check
0,What is Amazon S3 used for?,1,PASS - Source returned
1,What is Amazon EC2?,1,PASS - Source returned
2,What is AWS IAM used for?,1,PASS - Source returned
3,How can I back up data in Amazon S3?,1,PASS - Source returned
4,Who won the FIFA World Cup?,2,PASS - Out-of-scope handled


In [28]:

def inspect_retrieval(question):
    query_embedding = embedding_model.encode(
        [question]
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3,
        include=["documents", "metadatas", "distances"]
    )

    print("QUESTION:", question)
    print("=" * 70)

    for i, (document, metadata, distance) in enumerate(
        zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        ),
        start=1
    ):
        print(f"\nDocument {i}")
        print("Distance:", round(distance, 4))
        print("Source:", metadata.get("url", metadata.get("source", "Unknown")))
        print("Text preview:", document[:300].replace("\n", " "))

    print("\n" + "=" * 70)


inspect_retrieval("What is Amazon S3 used for?")
inspect_retrieval("Who won the FIFA World Cup?")


QUESTION: What is Amazon S3 used for?

Document 1
Distance: 0.5448
Source: https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html
Text preview: What is Amazon S3? - Amazon Simple Storage Service View a markdown version of this page What is Amazon S3? - Amazon Simple Storage Service Documentation Amazon Simple Storage Service (S3) User Guide Features of Amazon S3 How Amazon S3 works Amazon S3 data consistency model Related services Accessing

Document 2
Distance: 0.7135
Source: https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html
Text preview: GET, HEAD, and LIST requests to modify and process data as 						it is returned to an application. Filter rows, dynamically resize images, 						redact confidential data, and much more. Event 							notifications â Trigger workflows that use Amazon Simple Notification Service 						(Amazon SNS), A

Document 3
Distance: 0.7346
Source: https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html
Text preview:  which

In [31]:
test_questions = [
    "What is Amazon S3 used for?",
    "What is Amazon EC2?",
    "How does AWS IAM manage access?",
    "How can I back up data using AWS S3?",
    "What is the capital of France?"
]

for question in test_questions:
    answer, sources = ask_aws_assistant(question)

    print("Question:", question)
    print("Answer:", answer)
    print("Sources:", sources)
    print("-" * 60)

Question: What is Amazon S3 used for?
Answer: Based on the provided documentation, Amazon S3 is used to store and protect any amount of data for a range of use cases, such as:

* Data lakes
* Websites
* Mobile applications
* Backup and restore
* Archive
* Enterprise applications
* IoT devices
* Big data analytics
Sources: ['https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html']
------------------------------------------------------------
Question: What is Amazon EC2?
Answer: Amazon EC2 stands for Amazon Elastic Compute Cloud. It provides on-demand, scalable computing capacity in the AWS Cloud. Using Amazon EC2 helps reduce hardware costs so you can develop and deploy applications faster. You can use it to launch virtual servers, configure security and networking, and manage storage.
Sources: ['https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/concepts.html']
------------------------------------------------------------
Question: How does AWS IAM manage access?
Answer: Ba

In [32]:
test_questions = [
    ("What is Amazon S3 used for?", "S3"),
    ("What is Amazon EC2?", "EC2"),
    ("How does AWS IAM manage access?", "IAM"),
    ("How can I back up data using AWS S3?", "S3"),
    ("What is the capital of France?", "Unrelated")
]

results = []

for question, expected_topic in test_questions:
    answer, sources = ask_aws_assistant(question)

    if expected_topic == "Unrelated":
        status = (
            "PASS"
            if len(sources) == 0
            and "couldn't find sufficiently relevant" in answer.lower()
            else "CHECK"
        )
    else:
        matching_source = any(
            expected_topic.lower() in source.lower()
            for source in sources
        )
        status = "PASS" if matching_source else "CHECK"

    results.append({
        "Question": question,
        "Expected Topic": expected_topic,
        "Source Count": len(sources),
        "Status": status
    })

for result in results:
    print(result)

passed = sum(
    result["Status"] == "PASS"
    for result in results
)

print("\nEvaluation Summary")
print("Total questions:", len(results))
print("Passed:", passed)
print("Needs checking:", len(results) - passed)
print("Pass rate:", round(passed / len(results) * 100, 2), "%")

{'Question': 'What is Amazon S3 used for?', 'Expected Topic': 'S3', 'Source Count': 1, 'Status': 'PASS'}
{'Question': 'What is Amazon EC2?', 'Expected Topic': 'EC2', 'Source Count': 1, 'Status': 'PASS'}
{'Question': 'How does AWS IAM manage access?', 'Expected Topic': 'IAM', 'Source Count': 1, 'Status': 'PASS'}
{'Question': 'How can I back up data using AWS S3?', 'Expected Topic': 'S3', 'Source Count': 1, 'Status': 'PASS'}
{'Question': 'What is the capital of France?', 'Expected Topic': 'Unrelated', 'Source Count': 0, 'Status': 'PASS'}

Evaluation Summary
Total questions: 5
Passed: 5
Needs checking: 0
Pass rate: 100.0 %


In [33]:
import pandas as pd
from IPython.display import display

evaluation_df = pd.DataFrame(results)

evaluation_df.to_csv(
    "aws_rag_evaluation.csv",
    index=False
)

display(evaluation_df)

print("Evaluation report saved as aws_rag_evaluation.csv")

,Question,Expected Topic,Source Count,Status
0,What is Amazon S3 used for?,S3,1,PASS
1,What is Amazon EC2?,EC2,1,PASS
2,How does AWS IAM manage access?,IAM,1,PASS
3,How can I back up data using AWS S3?,S3,1,PASS
4,What is the capital of France?,Unrelated,0,PASS


Evaluation report saved as aws_rag_evaluation.csv


In [ ]:
import gradio as gr

def chat_with_aws(question, history):
    if not question.strip():
        return "Please enter an AWS question."

    try:
        answer, sources = ask_aws_assistant(question)

        if sources:
            source_text = "\n".join(
                f"- {source}" for source in sources
            )
            return f"{answer}\n\nSources:\n{source_text}"

        return answer

    except Exception as e:
        return f"An error occurred: {str(e)}"


demo = gr.ChatInterface(
    fn=chat_with_aws,
    title="AWS Cloud Support Assistant",
    description=(
        "Ask questions about Amazon S3, Amazon EC2, and AWS IAM. "
        "Answers are grounded in retrieved AWS documentation."
    ),
    examples=[
        "What is Amazon S3 used for?",
        "What is Amazon EC2?",
        "How does AWS IAM manage access?",
        "How can I back up data using AWS S3?",
        "What is the capital of France?"
    ]
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9a497ac6fb84c0c4df.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
